# Archive Current Analysis

This notebook archives the current analysis dataset and associated files into a date-stamped folder.

**What gets archived:**
- All processed data files (parquet files)
- All figure outputs (CSV summaries and plots)
- Current versions of all analysis notebooks
- README.md
- requirements.txt
- Metadata YAML files (if present)

**After archiving:**
- Cleans up the data folder to prepare for new analysis
- Preserves the raw data structure for reference

**⚠️ Warning:** Review the archive structure before running the cleanup step!

In [ ]:
import pandas as pd
import shutil
from pathlib import Path
from datetime import datetime
import json

# Define paths
project_root = Path.cwd().parent
data_dir = project_root / 'data'
notebooks_dir = project_root / 'notebooks'

# Create date stamp for archive (YYMMDD format)
date_stamp = datetime.now().strftime('%y%m%d')
archive_name = f"archive_{date_stamp}"
archive_dir = data_dir / archive_name

print(f"Archive directory: {archive_dir}")
print(f"Date stamp: {date_stamp}")

## Step 1: Create Archive Directory Structure

Creates the archive folder with subdirectories for different file types.

In [ ]:
# Create archive directory structure
archive_subdirs = {
    'processed': archive_dir / 'processed',
    'figures': archive_dir / 'figures',
    'notebooks': archive_dir / 'notebooks',
    'metadata': archive_dir / 'metadata',
    'config': archive_dir / 'config'
}

# Create all subdirectories
for subdir_name, subdir_path in archive_subdirs.items():
    subdir_path.mkdir(parents=True, exist_ok=True)
    print(f"Created: {subdir_path.relative_to(project_root)}")

print(f"\n✓ Archive structure created at: {archive_dir.relative_to(project_root)}")

## Step 2: Archive Processed Data Files

Copies all parquet files from the processed data folder.

In [ ]:
# Archive processed data files
processed_dir = data_dir / 'processed'
archived_files = []

if processed_dir.exists():
    for file_path in processed_dir.glob('*.parquet'):
        dest_path = archive_subdirs['processed'] / file_path.name
        shutil.copy2(file_path, dest_path)
        archived_files.append(file_path.name)
        print(f"Archived: {file_path.name}")
    
    print(f"\n✓ Archived {len(archived_files)} processed data files")
else:
    print("⚠️  No processed data directory found")

## Step 3: Archive Figure Outputs

Copies all CSV summaries and plots from the figures folder.

In [ ]:
# Archive figures and summary data
figures_dir = data_dir / 'figures'
archived_figures = []

if figures_dir.exists():
    for file_path in figures_dir.iterdir():
        if file_path.is_file():
            dest_path = archive_subdirs['figures'] / file_path.name
            shutil.copy2(file_path, dest_path)
            archived_figures.append(file_path.name)
            print(f"Archived: {file_path.name}")
    
    print(f"\n✓ Archived {len(archived_figures)} figure/summary files")
else:
    print("⚠️  No figures directory found")

## Step 4: Archive Notebooks

Copies all analysis notebooks to preserve the exact code used for this analysis.

In [ ]:
# Archive all notebooks
archived_notebooks = []

if notebooks_dir.exists():
    for notebook_path in notebooks_dir.glob('*.ipynb'):
        dest_path = archive_subdirs['notebooks'] / notebook_path.name
        shutil.copy2(notebook_path, dest_path)
        archived_notebooks.append(notebook_path.name)
        print(f"Archived: {notebook_path.name}")
    
    print(f"\n✓ Archived {len(archived_notebooks)} notebooks")
else:
    print("⚠️  No notebooks directory found")

## Step 5: Archive Metadata Files

Copies YAML metadata files if they exist.

In [ ]:
# Archive metadata files (YAML)
metadata_dir = data_dir / 'metadata'
archived_metadata = []

if metadata_dir.exists():
    for file_path in metadata_dir.glob('*.yaml'):
        dest_path = archive_subdirs['metadata'] / file_path.name
        shutil.copy2(file_path, dest_path)
        archived_metadata.append(file_path.name)
        print(f"Archived: {file_path.name}")
    
    # Also copy .yml files
    for file_path in metadata_dir.glob('*.yml'):
        dest_path = archive_subdirs['metadata'] / file_path.name
        shutil.copy2(file_path, dest_path)
        archived_metadata.append(file_path.name)
        print(f"Archived: {file_path.name}")
    
    print(f"\n✓ Archived {len(archived_metadata)} metadata files")
else:
    print("ℹ️  No metadata directory found (optional)")

## Step 6: Archive Configuration Files

Copies README.md and requirements.txt for reproducibility.

In [ ]:
# Archive configuration and documentation files
config_files = [
    project_root / 'README.md',
    project_root.parent / 'requirements.txt'  # requirements.txt is one level up
]

archived_configs = []
for file_path in config_files:
    if file_path.exists():
        dest_path = archive_subdirs['config'] / file_path.name
        shutil.copy2(file_path, dest_path)
        archived_configs.append(file_path.name)
        print(f"Archived: {file_path.name}")
    else:
        print(f"⚠️  Not found: {file_path.name}")

print(f"\n✓ Archived {len(archived_configs)} configuration files")

## Step 7: Create Archive Manifest

Creates a manifest file documenting what was archived and when.

In [ ]:
# Create archive manifest
manifest = {
    'archive_date': datetime.now().isoformat(),
    'archive_name': archive_name,
    'archived_files': {
        'processed_data': archived_files,
        'figures': archived_figures,
        'notebooks': archived_notebooks,
        'metadata': archived_metadata,
        'config': archived_configs
    },
    'total_files': len(archived_files) + len(archived_figures) + len(archived_notebooks) + len(archived_metadata) + len(archived_configs)
}

# Save manifest as JSON
manifest_path = archive_dir / 'MANIFEST.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

# Also create a human-readable README for the archive
readme_content = f"""# Archive {archive_name}

**Created:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Total Files:** {manifest['total_files']}

## Contents

### Processed Data ({len(archived_files)} files)
{chr(10).join(f'- {f}' for f in archived_files)}

### Figures ({len(archived_figures)} files)
{chr(10).join(f'- {f}' for f in archived_figures)}

### Notebooks ({len(archived_notebooks)} files)
{chr(10).join(f'- {f}' for f in archived_notebooks)}

### Metadata ({len(archived_metadata)} files)
{chr(10).join(f'- {f}' for f in archived_metadata) if archived_metadata else '- None'}

### Configuration ({len(archived_configs)} files)
{chr(10).join(f'- {f}' for f in archived_configs)}

## Reproducibility

To reproduce this analysis:
1. Install dependencies from `config/requirements.txt`
2. Review the notebooks in `notebooks/` in order (00, 01, 02)
3. Processed data is available in `processed/`
4. Figure outputs are in `figures/`
"""

readme_path = archive_dir / 'README.md'
with open(readme_path, 'w') as f:
    f.write(readme_content)

print(f"✓ Created archive manifest: {manifest_path.relative_to(project_root)}")
print(f"✓ Created archive README: {readme_path.relative_to(project_root)}")
print(f"\n{'='*60}")
print(f"ARCHIVE COMPLETE: {archive_dir.relative_to(project_root)}")
print(f"Total files archived: {manifest['total_files']}")
print(f"{'='*60}")

## Step 8: Verify Archive Contents

Review the archive structure before proceeding to cleanup.

In [ ]:
# Display archive structure
print("Archive Structure:")
print(f"\n{archive_dir.relative_to(project_root)}/")

for subdir_name, subdir_path in sorted(archive_subdirs.items()):
    file_count = len(list(subdir_path.iterdir()))
    print(f"  ├── {subdir_name}/ ({file_count} files)")
    
print(f"  ├── MANIFEST.json")
print(f"  └── README.md")

# Calculate total archive size
total_size = sum(f.stat().st_size for f in archive_dir.rglob('*') if f.is_file())
size_mb = total_size / (1024 * 1024)
print(f"\nTotal archive size: {size_mb:.2f} MB")

---

# ⚠️ CLEANUP SECTION - PROCEED WITH CAUTION

The following cells will remove files from the data directory to prepare for a new analysis.

**Review the archive above before proceeding!**

## Step 9: Clean Up Processed Data

Removes processed parquet files (now archived).

In [ ]:
# Clean up processed data
cleaned_processed = []

if processed_dir.exists():
    for file_path in processed_dir.glob('*.parquet'):
        file_path.unlink()
        cleaned_processed.append(file_path.name)
        print(f"Removed: {file_path.name}")
    
    print(f"\n✓ Removed {len(cleaned_processed)} processed data files")
else:
    print("No processed data to clean")

## Step 10: Clean Up Figures

Removes all figure outputs (now archived).

In [ ]:
# Clean up figures
cleaned_figures = []

if figures_dir.exists():
    for file_path in figures_dir.iterdir():
        if file_path.is_file():
            file_path.unlink()
            cleaned_figures.append(file_path.name)
            print(f"Removed: {file_path.name}")
    
    print(f"\n✓ Removed {len(cleaned_figures)} figure files")
else:
    print("No figures to clean")

## Step 11: Clean Up Metadata (Optional)

Optionally removes metadata files. **Note:** You may want to keep these for the next analysis.

In [ ]:
# Optional: Clean up metadata
# Uncomment the following code if you want to remove metadata files

CLEAN_METADATA = False  # Set to True to remove metadata files

cleaned_metadata = []

if CLEAN_METADATA and metadata_dir.exists():
    for file_path in metadata_dir.glob('*.yaml'):
        file_path.unlink()
        cleaned_metadata.append(file_path.name)
        print(f"Removed: {file_path.name}")
    
    for file_path in metadata_dir.glob('*.yml'):
        file_path.unlink()
        cleaned_metadata.append(file_path.name)
        print(f"Removed: {file_path.name}")
    
    print(f"\n✓ Removed {len(cleaned_metadata)} metadata files")
else:
    print("ℹ️  Metadata files preserved (set CLEAN_METADATA=True to remove)")

## Step 12: Summary

Final summary of archiving and cleanup operations.

In [ ]:
# Final summary
print("="*60)
print("ARCHIVE & CLEANUP SUMMARY")
print("="*60)
print(f"\nArchive Location: {archive_dir.relative_to(project_root)}")
print(f"Archive Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nFiles Archived:")
print(f"  - Processed data: {len(archived_files)}")
print(f"  - Figures: {len(archived_figures)}")
print(f"  - Notebooks: {len(archived_notebooks)}")
print(f"  - Metadata: {len(archived_metadata)}")
print(f"  - Config files: {len(archived_configs)}")
print(f"  - Total: {manifest['total_files']}")
print(f"\nFiles Removed:")
print(f"  - Processed data: {len(cleaned_processed)}")
print(f"  - Figures: {len(cleaned_figures)}")
print(f"  - Metadata: {len(cleaned_metadata)}")
print(f"\n✓ Ready for new analysis!")
print("="*60)

# Note about raw data
print("\n📝 Note: Raw data in 'data/raw/' has been preserved.")
print("   If you need to clean raw data folders, do this manually.")